In [ ]:
import time
import struct
import numpy as np
import scipy.ndimage
from pynq import Overlay, allocate

# Main input
ker_size =5
if_random = 0
inp_size = 128

if(ker_size == 3):
    ol = Overlay("design_1_wrapper.bit")
    print("Overlay loaded for design 1")
    print("IP blocks:", list(ol.ip_dict.keys()))
elif(ker_size == 5):
    ol = Overlay("design_2_wrapper.bit")
    print("Overlay loaded for design 2")
    print("IP blocks:", list(ol.ip_dict.keys()))

dma         = ol.axi_dma_0
conv_stream  = ol.conv2d_0
hw_timer    = ol.axi_timer_0

In [ ]:
kernel_size = ker_size
N_SAMPLES = inp_size * inp_size


if (if_random):
    # Random image
    img = np.random.randint(low=0, high=255, size=(inp_size, inp_size))/255
else:
    # Dog image
    from PIL import Image
    img_pil = Image.open("dog.jpg").convert("L")
    img_pil = img_pil.resize((inp_size, inp_size))
    img     = np.array(img_pil).astype(np.float32) / 255.0

if (if_random):
    # Random kernel
    kernel = np.random.uniform(-1, 1, (kernel_size, kernel_size))

else:
    # Sharpening kernel
    if(ker_size == 3):
        kernel = np.array([[ 0, -1,  0],
                           [-1,  5, -1],
                           [ 0, -1,  0]], dtype=np.float32)
    elif(ker_size == 5):
        kernel = np.array([[ 0,  0, -1,  0,  0],
                           [ 0, -1, -2, -1,  0],
                           [-1, -2, 16, -2, -1],
                           [ 0, -1, -2, -1,  0],
                           [ 0,  0, -1,  0,  0]], dtype=np.float32) / 4.0

t0 = time.perf_counter()
exp_out = scipy.ndimage.convolve(img, kernel, mode='constant', cval=0.0)
t_sw = time.perf_counter() - t0
exp_out = np.clip(exp_out, 0, 1)

print(f"Software: {N_SAMPLES} samples in {t_sw*1e3:.2f} ms "
      f"({t_sw/N_SAMPLES*1e6:.1f} us/sample)")

In [ ]:
def float_to_raw_int(f_val):

    scale = 1 << 12  # 2^12 = 4096
    raw = int(round(f_val * scale))
    
    # Clamp to 16-bit signed range
    raw = max(min(raw, 32767), -32768)
    
    # Convert to unsigned 16-bit representation (like hardware register)
    return raw & 0xFFFF

In [ ]:
TCSR0, TLR0, TCR0 = 0x00, 0x04, 0x08
FCLK_MHZ = 100.0

def timer_start(tmr):
    tmr.write(TLR0, 0)
    tmr.write(TCSR0, 0x020)   # load
    tmr.write(TCSR0, 0x080)   # enable, count up

def timer_stop(tmr):
    cycles = tmr.read(TCR0)
    tmr.write(TCSR0, 0x000)
    return cycles

In [ ]:
F = 12
scale = 1 << F

# Allocate as int32 to match 32-bit AXI stream
in_buf  = allocate(shape=(N_SAMPLES,), dtype=np.int32)
out_buf = allocate(shape=(N_SAMPLES,), dtype=np.int32)

# Cast fixed-point values to int32 (sign-extends correctly)
in_fixed = (img * scale).astype(np.int16).astype(np.int32)
np.copyto(in_buf, in_fixed.flatten())
out_buf[:] = 0

if(ker_size == 3):
    conv_stream.write(0x10, float_to_raw_int(kernel[0][0]))
    conv_stream.write(0x18, float_to_raw_int(kernel[0][1]))
    conv_stream.write(0x20, float_to_raw_int(kernel[0][2]))

    conv_stream.write(0x28, float_to_raw_int(kernel[1][0]))
    conv_stream.write(0x30, float_to_raw_int(kernel[1][1]))
    conv_stream.write(0x38, float_to_raw_int(kernel[1][2]))

    conv_stream.write(0x40, float_to_raw_int(kernel[2][0]))
    conv_stream.write(0x48, float_to_raw_int(kernel[2][1]))
    conv_stream.write(0x50, float_to_raw_int(kernel[2][2]))
    
elif(ker_size == 5):
    conv_stream.write(0x10, float_to_raw_int(kernel[0][0]))
    conv_stream.write(0x18, float_to_raw_int(kernel[0][1]))
    conv_stream.write(0x20, float_to_raw_int(kernel[0][2]))
    conv_stream.write(0x28, float_to_raw_int(kernel[0][3]))
    conv_stream.write(0x30, float_to_raw_int(kernel[0][4]))

    conv_stream.write(0x38, float_to_raw_int(kernel[1][0]))
    conv_stream.write(0x40, float_to_raw_int(kernel[1][1]))
    conv_stream.write(0x48, float_to_raw_int(kernel[1][2]))
    conv_stream.write(0x50, float_to_raw_int(kernel[1][3]))
    conv_stream.write(0x58, float_to_raw_int(kernel[1][4]))

    conv_stream.write(0x60, float_to_raw_int(kernel[2][0]))
    conv_stream.write(0x68, float_to_raw_int(kernel[2][1]))
    conv_stream.write(0x70, float_to_raw_int(kernel[2][2]))
    conv_stream.write(0x78, float_to_raw_int(kernel[2][3]))
    conv_stream.write(0x80, float_to_raw_int(kernel[2][4]))

    conv_stream.write(0x88, float_to_raw_int(kernel[3][0]))
    conv_stream.write(0x90, float_to_raw_int(kernel[3][1]))
    conv_stream.write(0x98, float_to_raw_int(kernel[3][2]))
    conv_stream.write(0xA0, float_to_raw_int(kernel[3][3]))
    conv_stream.write(0xA8, float_to_raw_int(kernel[3][4]))

    conv_stream.write(0xB0, float_to_raw_int(kernel[4][0]))
    conv_stream.write(0xB8, float_to_raw_int(kernel[4][1]))
    conv_stream.write(0xC0, float_to_raw_int(kernel[4][2]))
    conv_stream.write(0xC8, float_to_raw_int(kernel[4][3]))
    conv_stream.write(0xD0, float_to_raw_int(kernel[4][4]))

# Write image size (correct address)
conv_stream.write(0xd8, inp_size)

# Start IP
conv_stream.write(0x00, 0x01)

t0 = time.perf_counter()

# IMPORTANT: recv first to avoid deadlock
# print("recv setup")
timer_start(hw_timer)
dma.recvchannel.transfer(out_buf)

# print("send setup")
dma.sendchannel.transfer(in_buf)

# print("send wait")
dma.sendchannel.wait()

# print("recv wait")
dma.recvchannel.wait()
cycles = timer_stop(hw_timer)

# t_dma = time.perf_counter() - t0
y_fixed = np.array(out_buf, dtype=np.int32).astype(np.int16)  # truncate back to 16-bit
y_dma   = y_fixed.astype(np.float32) / scale

# Clipping values
y_dma = np.clip(y_dma, 0, 1)


total_us = cycles / FCLK_MHZ
per_sample_us = total_us / N_SAMPLES

print(f"HW timer: {cycles} cycles = {total_us:.1f} us total @ {FCLK_MHZ:.0f} MHz")
print(f"HW: {N_SAMPLES} samples in {total_us/1e3:.2f} ms ({per_sample_us:.1f} us/sample)")

# print(f"DMA: {N_SAMPLES} samples in {t_dma*1e3:.2f} ms "
#       f"({t_dma/N_SAMPLES*1e6:.1f} us/sample)")

In [ ]:
print("y_dma:", y_dma)

In [ ]:
np.max(np.abs(y_dma - exp_out.flatten()))

In [ ]:
y_dma.shape

In [ ]:
128 * 128

In [ ]:
np.any(y_dma>1)

In [ ]:
import matplotlib.pyplot as plt

if(if_random == 0):
    # Reshape outputs back to 2D
    img_display = img  # already (128, 128), float
    hw_out_2d   = y_dma.reshape(inp_size, inp_size)
    sw_out_2d   = exp_out  # scipy reference output

    # Difference map
    diff = np.abs(hw_out_2d - sw_out_2d)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))

    axes[0].imshow(img_display, cmap='gray')
    axes[0].set_title('Input Image')
    axes[0].axis('off')

    axes[1].imshow(hw_out_2d, cmap='gray')
    axes[1].set_title('HW Output (FPGA)')
    axes[1].axis('off')

    axes[2].imshow(sw_out_2d, cmap='gray')
    axes[2].set_title('SW Output (SciPy)')
    axes[2].axis('off')

    im = axes[3].imshow(diff, cmap='hot_r')
    axes[3].set_title(f'|HW - SW| (max={diff.max():.4f})')
    axes[3].axis('off')
    plt.colorbar(im, ax=axes[3])

    plt.tight_layout()
    plt.savefig("conv_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Max error : {diff.max():.6f}")
    print(f"Mean error: {diff.mean():.6f}")
    print(f"MSE       : {(diff**2).mean():.8f}")